SFT → GRPO: обычный и динамический отбор при одинаковом бюджете генерации.

GPU T4 ×2, Internet, прежний Spider/BIRD, SFT adapter/ + run.json + evaluation/ и COMET_API_KEY. Исходный SFT имеет accuracy 71.03%.

Оба варианта начинают от исходного SFT, LR 2.5e-6, temperature 0.8, 4 ответа на вопрос, beta=0, thinking off. Доступен весь train. На каждый вариант выделено 2048 вопросов × 4 ответа. Поток вопросов одинаков и проверяется по сохранённым ID. Число токенов и GPU-время учитываются отдельно.

В динамическом варианте до обновления весов добираются группы с различающимися наградами. Максимум 8 раундов добора на обновление. При нехватке подходящих групп используются группы с нулевым advantage. Число обновлений у вариантов различается.

Сначала автоматически выполняется GPU-smoke на 8 вопросах. Затем два обучения запускаются параллельно, экспорт и полная validation выполняются последовательно. Ориентир по предыдущим логам — 4–5 часов для длинного контрольного обучения, затем оценка. При ошибке доступные артефакты сохраняются в ZIP. Снимки адаптеров не содержат состояние оптимизатора для точного resume.


In [ ]:
%pip install "vllm==0.31.0" "transformers==5.17.0" "peft==0.21.2" "accelerate==1.15.0" "comet-ml==3.59.0" python-dotenv
%pip install --no-deps --index-url https://download.pytorch.org/whl/cu130 "torchaudio==2.11.0+cu130"
%pip uninstall -y torchao torchcodec
%pip install --no-deps "trl==1.16.0"
%pip install --no-deps --no-cache-dir --index-url https://download.pytorch.org/whl "nvidia-cudnn-cu13==9.23.0.39"

In [ ]:
import os
import sys
import json
import shutil
import subprocess
from pathlib import Path
from zipfile import ZipFile, ZIP_DEFLATED
from datetime import datetime, timezone

WORK = Path("/kaggle/working")
INPUT = Path("/kaggle/input")
REPO = WORK / "text2sql-grpo-sft"
RUN = WORK / "text2sql_runs" / "v1"
BASE = "Qwen/Qwen3.5-2B"
REVISION = "15852e8c16360a2fea060d615a32b45270f8a8fc"

if (REPO / ".git").exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only", "origin", "main"], check=True)
else:
    subprocess.run(["git", "clone", "https://github.com/dancravik/text2sql-grpo.git", str(REPO)], check=True)

DATA_DIR = None  # При необходимости укажите Path("/kaggle/input/.../...")
if DATA_DIR is None:
    roots = sorted({
        p.parents[1] for p in INPUT.rglob("train_spider.json")
        if p.parent.name == "spider"
        and (p.parents[1] / "minidev/MINIDEV/mini_dev_sqlite.json").is_file()
    })
    if len(roots) == 1:
        DATA_DIR = roots[0]
    elif roots:
        raise ValueError(f"Укажите DATA_DIR вручную: {roots}")
    else:
        DATA_DIR = WORK / "text2sql_data"
        if not (DATA_DIR / "spider/train_spider.json").is_file():
            archives = list(INPUT.rglob("text2sql-data.zip"))
            if len(archives) != 1:
                raise ValueError(f"Добавьте прежний датасет или задайте DATA_DIR: {archives}")
            with ZipFile(archives[0]) as archive:
                archive.extractall(DATA_DIR)

RUN.mkdir(parents=True, exist_ok=True)
os.environ["DATA_DIR"] = str(DATA_DIR)
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ.pop("TORCH_CUDNN_V8_API_DISABLED", None)
subprocess.run([sys.executable, "scripts/freeze_splits.py", "--verify"], cwd=REPO, check=True)
print("DATA_DIR:", DATA_DIR)
print("Свободно GiB:", round(shutil.disk_usage(WORK).free / 1024**3, 2))

In [ ]:
from kaggle_secrets import UserSecretsClient

os.environ["COMET_API_KEY"] = UserSecretsClient().get_secret("COMET_API_KEY")
os.environ["COMET_WORKSPACE"] = "danchenko-rtm"
os.environ["COMET_PROJECT_NAME"] = "text2sql-grpo"

subprocess.run([sys.executable, "-u", "-c", r"""
import importlib.metadata
import torch
from transformers import Trainer
from peft import LoraConfig
from vllm import LLM
from trl import GRPOConfig, GRPOTrainer
for package in ("torch", "transformers", "peft", "accelerate", "vllm", "comet-ml", "nvidia-cudnn-cu13", "trl"):
    print(package, importlib.metadata.version(package))
assert torch.cuda.device_count() == 1
x = torch.randn(1, 6144, 512, device="cuda", dtype=torch.float16, requires_grad=True)
w = torch.randn(6144, 1, 4, device="cuda", dtype=torch.float16)
torch.nn.functional.conv1d(x, w, groups=6144, padding=3).float().mean().backward()
torch.cuda.synchronize()
print("GPU и FP16 conv1d forward/backward OK")
from comet_tracking import start_experiment
experiment = start_experiment(name="sft_grpo_sampling_environment_check")
try:
    experiment.log_metric("environment_ready", 1)
finally:
    experiment.end()
"""], cwd=REPO, env={**os.environ, "PYTHONPATH": str(REPO / "src")}, check=True)
gpu_ids = subprocess.check_output(["nvidia-smi", "--query-gpu=index", "--format=csv,noheader"], text=True).split()
assert len(gpu_ids) >= 2, f"Нужны две GPU, найдено {gpu_ids}"
print("GPU для независимых пилотов:", gpu_ids[:2])

Восстановление сохранённого SFT. При нескольких копиях укажите SFT_INPUT как путь к нужной папке в /kaggle/input. Эта ячейка проверяет отчёт и SHA256 весов. Подробная проверка стартовых тензоров выполняется обучающим процессом.

In [ ]:
import hashlib

SFT_INPUT = None  # Например Path("/kaggle/input/datasets/your-name/sft-model")
search_root = Path(SFT_INPUT) if SFT_INPUT is not None else INPUT

def find_sft(root):
    matches = []
    for report_path in root.rglob("run.json"):
        adapter = report_path.parent / "adapter/adapter_model.safetensors"
        if not adapter.is_file():
            continue
        report = json.loads(report_path.read_text())
        if (report.get("status") == "ok" and report.get("num_train_examples") == 5819
                and report.get("global_step") == 728
                and report.get("base_revision") == REVISION):
            matches.append(report_path.parent)
    return sorted(set(matches))

candidates = find_sft(search_root)
if not candidates:
    archives = sorted(search_root.rglob("sft_full_20261121_103258_artifacts.zip"))
    if len(archives) != 1:
        raise ValueError("Добавьте SFT adapter/ и run.json через Add Input. "
                         f"Найдено подходящих ZIP: {archives}")
    restored = WORK / "restored_sft_input"
    restored.mkdir(exist_ok=True)
    with ZipFile(archives[0]) as archive:
        for member in archive.infolist():
            if not (restored / member.filename).resolve().is_relative_to(restored.resolve()):
                raise ValueError("Некорректный путь в ZIP")
        archive.extractall(restored)
    candidates = find_sft(restored)
if len(candidates) != 1:
    raise ValueError(f"Укажите SFT_INPUT с одной копией полного SFT: {candidates}")

SFT = candidates[0]
report = json.loads((SFT / "run.json").read_text())
manifest_hash = hashlib.sha256((REPO / "splits/v1.json").read_bytes()).hexdigest()
assert report["manifest_sha256"] == manifest_hash
assert report["args"]["model"] == BASE and report["optimizer_steps_skipped"] == 0
weights_hash = hashlib.sha256((SFT / "adapter/adapter_model.safetensors").read_bytes()).hexdigest()
print("SFT:", SFT)
print("SHA256 адаптера:", weights_hash)
print("Шагов SFT:", report["global_step"])
if not (SFT / "evaluation/validation.jsonl").is_file():
    raise FileNotFoundError("Входной архив должен содержать evaluation/validation.jsonl для сравнения с SFT")

GROUP = "sft_grpo_sampling_" + datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")
BUNDLE = RUN / GROUP
print("Новый запуск:", GROUP)

Запуск всего сравнения. Сначала проверяется новый путь генерации и отбора. Итоговая comparison.json содержит сопоставление с SFT и попарное сопоставление двух способов отбора. Последнюю ячейку скачивания можно выполнить после ошибки.


In [ ]:
subprocess.run([
    sys.executable, "-u", "scripts/run_grpo_sampling.py",
    "--out-dir", str(BUNDLE), "--init-sft-run", str(SFT),
    "--gpus", *gpu_ids[:2], "--rollout-budget", "2048",
], cwd=REPO, env=os.environ.copy(), check=True)


Скачивание архива. Здесь сохраняются уже доступные результаты, включая логи неудачного этапа. При полном завершении таблица выводит точность и затраты обоих вариантов.


In [ ]:
from IPython.display import FileLink, display

ARCHIVE = BUNDLE.parent / f"{BUNDLE.name}_complete.zip"
if (BUNDLE / "comparison.json").is_file():
    result = json.loads((BUNDLE / "comparison.json").read_text())
    print("SFT:", f"{result['sft_accuracy']:.2%}")
    for mode, row in result["results"].items():
        print(mode, f"accuracy={row['accuracy']:.2%}",
              f"шагов={row['optimizer_steps']}",
              f"GPU-часов={row['gpu_hours']:.2f}",
              f"токенов генерации={row['generated_completion_tokens']}")
    print("Dynamic против ordinary:", result["dynamic_vs_ordinary"])
if not ARCHIVE.is_file():
    raise FileNotFoundError(f"Архив пока не создан: {ARCHIVE}")
print("Архив:", ARCHIVE, f"{ARCHIVE.stat().st_size / 1024**2:.1f} МБ")
display(FileLink(ARCHIVE.relative_to(WORK).as_posix()))
